# Persist Read Server Examples

Examples for reading locally mirrored `uniform_orders` from the persist read server on this machine.

Sources configured here:

- `okex-intra-arb01`
- `binance-intra-arb01`
- `bybit-intra-arb01`


In [1]:
import time
import urllib.parse
import urllib.request
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc

BASE_URL = 'http://127.0.0.1:8822'
SOURCES = ['okex-intra-arb01', 'binance-intra-arb01', 'bybit-intra-arb01']
TABLE = 'uniform_orders'


## Helpers


In [2]:
def get_json(path, params=None, timeout=30):
    url = BASE_URL + path
    if params:
        url += '?' + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return __import__('json').loads(resp.read().decode('utf-8'))


def read_arrow(table, source_id, start_us, end_us, columns=None, timeout=120):
    params = {
        'table': table,
        'source_id': source_id,
        'start_us': int(start_us),
        'end_us': int(end_us),
        'format': 'arrow_ipc',
    }
    if columns:
        params['columns'] = ','.join(columns) if isinstance(columns, (list, tuple)) else columns
    url = BASE_URL + '/v1/read?' + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        data = resp.read()
    return ipc.open_stream(pa.BufferReader(data)).read_all()


def read_uniform_orders(source_id, start_us, end_us, columns=None):
    default_columns = [
        'ts_us', 'symbol', 'client_order_id', 'trading_venue',
        'side', 'price', 'amount_init', 'amount_update', 'status',
    ]
    table = read_arrow(TABLE, source_id, start_us, end_us, columns or default_columns)
    return table.to_pandas()


def utc_window(hours=1):
    end_us = int(time.time() * 1_000_000)
    start_us = end_us - int(hours * 3600 * 1_000_000)
    return start_us, end_us


def fmt_us(ts_us):
    return datetime.fromtimestamp(ts_us / 1_000_000, timezone.utc).isoformat()


## Health And Schema


In [3]:
print(get_json('/healthz'))
for source in SOURCES:
    schema = get_json('/v1/schema', {'table': TABLE, 'source_id': source})
    print(source, schema['table'], len(schema['columns']), schema['formats'])


{'ok': True}
okex-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']
binance-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']
bybit-intra-arb01 uniform_orders 22 ['arrow_ipc', 'parquet']


## Latest 1 Hour Row Counts


In [4]:
start_us, end_us = utc_window(hours=1)
print('UTC window:', fmt_us(start_us), '->', fmt_us(end_us))

summary = []
for source in SOURCES:
    df = read_uniform_orders(source, start_us, end_us, columns=['ts_us', 'symbol', 'client_order_id', 'trading_venue', 'status'])
    summary.append({'source_id': source, 'rows': len(df)})
pd.DataFrame(summary)


UTC window: 2026-06-11T04:07:27.626831+00:00 -> 2026-06-11T05:07:27.626831+00:00


,source_id,rows
0,okex-intra-arb01,4103
1,binance-intra-arb01,10
2,bybit-intra-arb01,2385


## Pull Each Source


In [5]:
okex_orders = read_uniform_orders('okex-intra-arb01', start_us, end_us)
okex_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781150848572405,ETHUSDT,2841939551495127041,OkexMargin,BUY,1652.60,0.03,0.030,FILLED
1,1781150848574224,ETHUSDT,2841939555790094337,OkexMargin,BUY,1652.48,0.03,0.000,CANCELED
2,1781150848576398,ETHUSDT,2840696729693586429,OkexFutures,SELL,0.00,0.03,0.000,NEW
3,1781150848576412,ETHUSDT,2841939560085061633,OkexMargin,BUY,1652.37,0.03,0.000,CANCELED
4,1781150848576693,ETHUSDT,2840696729693586429,OkexFutures,SELL,1651.63,0.03,0.004,PARTIALLY_FILLED


In [6]:
binance_orders = read_uniform_orders('binance-intra-arb01', start_us, end_us)
binance_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781152946046462,KNCUSDT,2629588620259885057,BinanceMargin,SELL,0.1223,408.0,84.1,PARTIALLY_FILLED
1,1781153097979094,KNCUSDT,2629588620259885057,BinanceMargin,SELL,0.1223,408.0,323.9,FILLED
2,1781153097982477,KNCUSDT,2628931292695101516,BinanceFutures,BUY,0.0000,408.0,0.0,NEW
3,1781153097984229,KNCUSDT,2628931292695101516,BinanceFutures,BUY,0.1222,408.0,408.0,FILLED
4,1781153445615986,KNCUSDT,2629362984152989697,BinanceMargin,SELL,0.1227,407.0,407.0,FILLED


In [7]:
bybit_orders = read_uniform_orders('bybit-intra-arb01', start_us, end_us)
bybit_orders.head()


,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status
0,1781150853186110,RENDERUSDT,3061383964817096705,BybitMargin,BUY,1.5550,32.1,0.0,NEW
1,1781150854732868,SPXUSDT,3061383758658666497,BybitMargin,BUY,0.3207,155.0,0.0,CANCELED
2,1781150854865493,ICPUSDT,3061384012061736961,BybitMargin,SELL,2.2870,21.8,0.0,NEW
3,1781150857129438,AVAXUSDT,3061384063601344513,BybitMargin,BUY,6.5760,7.6,0.0,NEW
4,1781150857436761,TONUSDT,3061383724298928129,BybitMargin,BUY,1.6510,30.2,0.0,CANCELED


## Combined Analysis Example


In [8]:
frames = []
for source in SOURCES:
    df = read_uniform_orders(source, start_us, end_us)
    df.insert(0, 'source_id', source)
    frames.append(df)
orders = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
orders['ts'] = pd.to_datetime(orders['ts_us'], unit='us', utc=True)
orders.head()


,source_id,ts_us,symbol,client_order_id,trading_venue,side,price,amount_init,amount_update,status,ts
0,okex-intra-arb01,1781150848572405,ETHUSDT,2841939551495127041,OkexMargin,BUY,1652.60,0.03,0.030,FILLED,2026-06-11 04:07:28.572405+00:00
1,okex-intra-arb01,1781150848574224,ETHUSDT,2841939555790094337,OkexMargin,BUY,1652.48,0.03,0.000,CANCELED,2026-06-11 04:07:28.574224+00:00
2,okex-intra-arb01,1781150848576398,ETHUSDT,2840696729693586429,OkexFutures,SELL,0.00,0.03,0.000,NEW,2026-06-11 04:07:28.576398+00:00
3,okex-intra-arb01,1781150848576412,ETHUSDT,2841939560085061633,OkexMargin,BUY,1652.37,0.03,0.000,CANCELED,2026-06-11 04:07:28.576412+00:00
4,okex-intra-arb01,1781150848576693,ETHUSDT,2840696729693586429,OkexFutures,SELL,1651.63,0.03,0.004,PARTIALLY_FILLED,2026-06-11 04:07:28.576693+00:00


In [9]:
orders.groupby(['source_id', 'status']).size().reset_index(name='orders').sort_values(['source_id', 'orders'], ascending=[True, False])


,source_id,status,orders
0,binance-intra-arb01,FILLED,6
1,binance-intra-arb01,NEW,3
2,binance-intra-arb01,PARTIALLY_FILLED,1
3,bybit-intra-arb01,CANCELED,1281
6,bybit-intra-arb01,NEW,1028
5,bybit-intra-arb01,FILLED,69
7,bybit-intra-arb01,PARTIALLY_FILLED,5
4,bybit-intra-arb01,EXPIRED,2
10,okex-intra-arb01,NEW,2048
8,okex-intra-arb01,CANCELED,2006


In [10]:
orders.groupby(['source_id', 'symbol']).size().reset_index(name='orders').sort_values('orders', ascending=False).head(20)


,source_id,symbol,orders
71,okex-intra-arb01,ETHUSDT,1656
72,okex-intra-arb01,SOLUSDT,1228
70,okex-intra-arb01,DOGEUSDT,612
69,okex-intra-arb01,BTCUSDT,555
23,bybit-intra-arb01,ETHUSDT,467
48,bybit-intra-arb01,RENDERUSDT,240
47,bybit-intra-arb01,POPCATUSDT,236
50,bybit-intra-arb01,SOLUSDT,167
3,bybit-intra-arb01,AEROUSDT,130
63,bybit-intra-arb01,WIFUSDT,112
